### Z ordering co locates similar data in same file

Finding databricks provided sample file for  z order demo

In [0]:
%fs
ls /databricks-datasets

path,name,size,modificationTime
dbfs:/databricks-datasets/COVID/,COVID/,0,1788165570151
dbfs:/databricks-datasets/README.md,README.md,976,1532502324000
dbfs:/databricks-datasets/Rdatasets/,Rdatasets/,0,1788165570151
dbfs:/databricks-datasets/SPARK_README.md,SPARK_README.md,3359,1455505834000
dbfs:/databricks-datasets/adult/,adult/,0,1788165570151
dbfs:/databricks-datasets/airlines/,airlines/,0,1788165570151
dbfs:/databricks-datasets/amazon/,amazon/,0,1788165570151
dbfs:/databricks-datasets/asa/,asa/,0,1788165570151
dbfs:/databricks-datasets/atlas_higgs/,atlas_higgs/,0,1788165570151
dbfs:/databricks-datasets/bikeSharing/,bikeSharing/,0,1788165570151


In [0]:
%fs
ls /databricks-datasets/nyctaxi/tables

path,name,size,modificationTime
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/,nyctaxi_yellow/,0,1788165661262


Chose NYC taxi demo table

In [0]:
%fs
ls /databricks-datasets/nyctaxi/tables/nyctaxi_yellow/

path,name,size,modificationTime
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/_delta_log/,_delta_log/,0,1788165691045
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-7dcc09ea-2fc1-491d-a234-3b0ce1db9336-c002.snappy.parquet,part-00000-7dcc09ea-2fc1-491d-a234-3b0ce1db9336-c002.snappy.parquet,374549044,1605327443000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-812df468-c3fb-405d-84d6-32cb249d8db9-c003.snappy.parquet,part-00000-812df468-c3fb-405d-84d6-32cb249d8db9-c003.snappy.parquet,189069652,1605327443000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-a4b4b3de-4231-4c0e-88d6-c14f202ff232-c000.snappy.parquet,part-00000-a4b4b3de-4231-4c0e-88d6-c14f202ff232-c000.snappy.parquet,373889711,1605327443000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-b3d2db79-4a87-4d35-8a20-a02725fb655f-c001.snappy.parquet,part-00000-b3d2db79-4a87-4d35-8a20-a02725fb655f-c001.snappy.parquet,376396848,1605327443000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-158e21e1-9d7b-44e9-ad15-3ba7e1797de8-c002.snappy.parquet,part-00001-158e21e1-9d7b-44e9-ad15-3ba7e1797de8-c002.snappy.parquet,364876497,1605327444000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9072b615-4da5-4c0c-b2c0-83f08d1944ac-c000.snappy.parquet,part-00001-9072b615-4da5-4c0c-b2c0-83f08d1944ac-c000.snappy.parquet,363016752,1605327465000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9baf2d71-1e4d-49a0-a131-03c825853637-c003.snappy.parquet,part-00001-9baf2d71-1e4d-49a0-a131-03c825853637-c003.snappy.parquet,203737885,1605327471000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-d61d36d1-b864-4dc2-b80b-92e7d1515232-c001.snappy.parquet,part-00001-d61d36d1-b864-4dc2-b80b-92e7d1515232-c001.snappy.parquet,365327633,1605327484000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00002-0e3744e2-5c15-4c60-baee-bd45cf02a686-c000.snappy.parquet,part-00002-0e3744e2-5c15-4c60-baee-bd45cf02a686-c000.snappy.parquet,355258373,1605327487000


In [0]:
%sql
describe formatted delta.`dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow`

col_name,data_type,comment
vendor_id,string,null
pickup_datetime,timestamp,null
dropoff_datetime,timestamp,null
passenger_count,int,null
trip_distance,double,null
pickup_longitude,double,null
pickup_latitude,double,null
rate_code_id,int,null
store_and_fwd_flag,string,null
dropoff_longitude,double,null


In [0]:
%sql
drop table orders_ext;
drop table if exists nyc_taxi;
create table nyc_taxi (
   vendor_id	string,
pickup_datetime	timestamp,
dropoff_datetime	timestamp,
passenger_count	int,
trip_distance	double,
pickup_longitude	double,
pickup_latitude	double,
rate_code_id	int,
store_and_fwd_flag	string,
dropoff_longitude	double,
dropoff_latitude	double,
payment_type	string,
fare_amount	double,
extra	double,
mta_tax	double,
tip_amount	double,
tolls_amount	double,
total_amount	double
) using delta
tblproperties (
delta.autoOptimize.optimizeWrite = false,
delta.autoOptimize.autoCompact = false
)
LOCATION 'abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat1'

Load the table with 10 parq files from above but split it to 200 files
Took only 10 files, are the whole table is huge. 

In [0]:
file_path = 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow'
all_files = [f.path for f in dbutils.fs.ls(file_path) if f.path.endswith('parquet')]

files10 = all_files[:10]
df = spark.read.parquet(*files10)
df1 = df.repartition(200)

df1.write.format('delta').mode('overwrite').saveAsTable("nyc_taxi")

90.8 million rows

In [0]:
%sql
select count (*) from nyc_taxi

count(1)
90848212


This query has to open about 132 files out of 200 to get the results. Pruning is very less

In [0]:
%sql
select count(*) from nyc_taxi where trip_distance > 100;

count(1)
209


Trip distance field is not stored in ordered manner , which makes the query in efficient. Do a ZORDER to optimize it

In [0]:
%sql
select min(trip_distance) ,max(trip_distance) , _metadata.file_name from nyc_taxi
group by _metadata.file_name order by  min(trip_distance);


min(trip_distance),max(trip_distance),file_name
0.0,100.0,part-00140-45994a2c-7d4c-44c1-b6eb-7e268c044341-c000.zstd.parquet
0.0,75.4,part-00118-8da54e56-8034-432e-bca6-1ff334326c20-c000.zstd.parquet
0.0,183.7,part-00155-631151c3-b9cf-44ae-8974-6925d982d06c-c000.zstd.parquet
0.0,100.0,part-00138-a7389ecf-cd57-4089-9437-b8ce64321a05-c000.zstd.parquet
0.0,107.1,part-00015-a294053f-9202-440c-8bbc-63ea3c25e77c-c000.zstd.parquet
0.0,166.0,part-00126-eae2d738-6467-4d2c-871e-cade9e0d1750-c000.zstd.parquet
0.0,756.7,part-00101-fd089d72-5c5f-4052-a4f9-0d1c94a09681-c000.zstd.parquet
0.0,183.1,part-00093-f4624446-609d-4f15-9d85-8bad2e9c31a7-c000.zstd.parquet
0.0,125.1,part-00184-279dda01-75ba-4d32-95ba-10f50a4153ed-c000.zstd.parquet
0.0,180.7,part-00129-1589c585-5132-4143-8bc3-8b6cf68a3dfb-c000.zstd.parquet


OPTIMIZE WITH  ZORDER

In [0]:
%sql
optimize nyc_taxi zorder by trip_distance

path,metrics
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat1,"List(11, 200, List(266583179, 409885150, 3.3251161390909094E8, 11, 3657627753), List(15538105, 15687987, 1.5592702825E7, 200, 3118540565), 0, List(minCubeSize(107374182400), List(0, 0), List(200, 3118540565), 0, List(200, 3118540565), 1, null), null, 0, 1, 200, 0, false, 0, 0, 1788170125637, 1788170522416, 4, 1, null, List(0, 0), null, 18, 18, 776321, 0, null)"


Optimize stats;
- numFilesAdded: 11
- numFilesRemoved: 200

Re run the query

In [0]:
%sql
select count(*) from nyc_taxi where trip_distance > 100;

count(1)
209


After Zorder
- number of files pruned	10
- number of files read	1
Just 1 file was needed to be read this time, after zorder

In [0]:
%sql
select min(trip_distance) ,max(trip_distance) , _metadata.file_name from nyc_taxi
group by _metadata.file_name order by  min(trip_distance);


min(trip_distance),max(trip_distance),file_name
0.0,0.66,part-00000-b27f6895-4354-469a-8004-b0880bdbdcc5-c000.snappy.parquet
0.66,0.9,part-00001-80338046-ac36-46cd-8b0c-035f478d14ac-c000.snappy.parquet
0.9,1.14,part-00002-db765be8-8ac6-4791-8f49-81a83d982df5-c000.snappy.parquet
1.14,1.4,part-00003-e13df4d1-3014-4483-af17-07eb68eeb66e-c000.snappy.parquet
1.4,1.7,part-00004-7313878c-bdc5-4b2f-8398-925652c45b60-c000.snappy.parquet
1.7,2.1,part-00005-a77e3ae4-e57e-473f-a90c-4faea5eeeacf-c000.snappy.parquet
2.1,2.55,part-00006-c4b2f34d-9553-4ca5-8a01-4715d874994d-c000.snappy.parquet
2.55,3.2,part-00007-59323698-d077-4c0d-8c8d-c21502a76c0f-c000.snappy.parquet
3.2,4.14,part-00008-d7a6c9df-d663-4aba-acd5-5fb0c64f40c9-c000.snappy.parquet
4.14,5.7,part-00009-b044fead-5ef8-45be-b5f3-b46b469db8c1-c000.snappy.parquet


After zorder each parquet file has unique range of trip distance values which are not overlapping